In [ ]:
import pandas as pd
import geopandas as gpd
from datetime import datetime
import numpy as np

In [ ]:
od_df = pd.read_csv("../../reports/deidentified_overdose_21022025.csv")
zipcode_gdf = gpd.read_file("../../data/zipcodes.geojson")

In [ ]:
# 2. Process the DeathDate to extract year and month
od_df["DeathDate"] = pd.to_datetime(od_df["DeathDate"])
od_df["YearMonth"] = od_df["DeathDate"].dt.strftime("%Y-%m")

# 3. Create a complete grid of all zipcodes × all year-months
# Get unique zipcodes and year-months
unique_zipcodes = zipcode_gdf["ZIPCODE"].unique()
unique_yearmonths = od_df["YearMonth"].unique()

# Create all possible combinations
grid = []
for zipcode in unique_zipcodes:
    for yearmonth in unique_yearmonths:
        grid.append({"ZIPCODE": zipcode, "YearMonth": yearmonth})

grid_df = pd.DataFrame(grid)

# 4. Group by zipcode and YearMonth, and aggregate drug counts and total count
drug_columns = [
    "Methamphetamine",
    "Heroin",
    "Cocaine",
    "Fentanyl",
    "Alcohol",
    "Prescription.opioids",
    "Any Opioids",
    "Benzodiazepines",
    "Others",
    "Any Drugs",
]

# Create aggregations dictionary for drug columns and age
agg_dict = {}
for drug in drug_columns:
    agg_dict[drug] = "sum"

agg_dict["Age"] = ["mean", "min", "max"]

# 5. Compute drug and age aggregations
agg_df = od_df.groupby(["ZIPCODE", "YearMonth"]).agg(agg_dict).reset_index()

# Fix column names (agg creates multiindex columns)
agg_df.columns = [
    f"{col[0]}_{col[1]}" if isinstance(col, tuple) else col for col in agg_df.columns
]

# 6. Add a total count column
count_df = (
    od_df.groupby(["ZIPCODE", "YearMonth"]).size().reset_index(name="TotalOverdoses")
)

# 7. Create race counts using a different approach
# Get unique race categories
race_categories = od_df["Race"].unique()
race_dfs = []

for race in race_categories:
    # For each race, create a DataFrame with count of that race by ZIPCODE and YearMonth
    race_count = (
        od_df[od_df["Race"] == race]
        .groupby(["ZIPCODE", "YearMonth"])
        .size()
        .reset_index()
    )
    race_count = race_count.rename(columns={0: f"Race_{race}"})
    race_dfs.append(race_count)

# 8. Create gender counts using the same approach
gender_categories = od_df["Gender"].unique()
gender_dfs = []

for gender in gender_categories:
    # For each gender, create a DataFrame with count of that gender by ZIPCODE and YearMonth
    gender_count = (
        od_df[od_df["Gender"] == gender]
        .groupby(["ZIPCODE", "YearMonth"])
        .size()
        .reset_index()
    )
    gender_count = gender_count.rename(columns={0: f"Gender_{gender}"})
    gender_dfs.append(gender_count)

# 9. Merge everything together, starting with the grid
result = grid_df

# Merge with drug and age aggregations
result = result.merge(agg_df, on=["ZIPCODE", "YearMonth"], how="left")
result = result.merge(count_df, on=["ZIPCODE", "YearMonth"], how="left")

# Merge with each race count
for race_df in race_dfs:
    result = result.merge(race_df, on=["ZIPCODE", "YearMonth"], how="left")

# Merge with each gender count
for gender_df in gender_dfs:
    result = result.merge(gender_df, on=["ZIPCODE", "YearMonth"], how="left")

# 10. Fill NaN values with 0
result = result.fillna(0)

# 11. Merge with spatial data
final_gdf = zipcode_gdf.merge(result, on="ZIPCODE", how="right")

# 12. Save as GeoPackage
final_gdf.to_file("overdose_timeseries.gpkg", driver="GPKG")